# 01 — Grouping, Aggregation and Pivot Tables (Pandas Intermediate)

**Level:** Intermediate | **Requirement:** pandas 3.x, numpy

this notebook pandas of all with zyada "interview + real job" which hissa is: **split-apply-combine**.
Excel me that work Pivot Table does is, pandas me wo `groupby` / `pivot_table` / `crosstab` do are —
but 100x zyada powerful and programmable.

## Is notebook me what seekhenge

- **Split-Apply-Combine** of concept (text diagram of sath)
- `groupby` object asal me what is (lazy is, some compute no does) — `.groups`, `.ngroups`, `get_group()`, iteration
- **Grouping of saare tareeqe**: column, multiple columns, Series/array, index level, function, dict/mapping, `pd.Grouper`
- `sort=False` (performance), `dropna=False` (NaN groups), `observed` (categoricals), `as_index=False` vs `reset_index()`
- **Aggregation**: single func, list of funcs, dict per column, lambda, **named aggregation**, custom funcs, `describe()` per group
- Built-in fast aggs: sum/mean/median/min/max/count/size/nunique/first/last/std/var/prod/quantile/idxmax
- **`count` vs `size`** of farq (NaN which kahani) — classic interview question
- **`transform`** (same-shape output): group mean fill, z-score, percent of total, rank, cumcount, cumsum, `ngroup`
- **`filter`** (poore group to to keep ya hatana), **`apply`** (when necessary, why slow, `group_keys`)
- Group inside window ops: `shift`, `diff`, `pct_change`, `rolling`, `expanding`, `head`/`tail`/`nth`
- **Top-N per group — 3 tareeqe** + timing comparison
- MultiIndex results, `unstack()` with pivot, **`pivot_table`** (margins/fill_value/aggfunc), **`crosstab`** (normalize)
- Business questions: revenue per region per month, AOV, conversion by channel, category share, best product per store, outlier flag
- Common mistakes, **LeetCode problems**, Interview Q&A, Practice exercises, Cheat-sheet

## Prerequisites
Pandas Basics: Series/DataFrame banana, `loc`/`iloc`, boolean filtering, `sort_values`, NaN of basic idea.

In [ ]:
# setup — sirf pandas and numpy (matplotlib is env me no is, plotting no will do)
import time                      # timing comparisons for
import numpy as np
import pandas as pd

# Display thora control — taake output readable 
pd.set_option("display.width", 110)          # console width
pd.set_option("display.max_columns", 25)     # columns chhupne not paayein

print("pandas:", pd.__version__)   # is notebook to pandas 3.x par written gaya is
print("numpy :", np.__version__)

## 1. Split-Apply-Combine — poora concept

every group-wise analysis 3 steps me is is:

1. **Split** — data to kisi key (ya keys) of hisab with tukdon me baant do.
2. **Apply** — every tukde par koi work do (sum, mean, custom function...).
3. **Combine** — natije to wapas ek DataFrame/Series me jod do.

```text
        RAW DATA                      SPLIT (by region)            APPLY (sum)      COMBINE
 region   amount
 ------   ------                   North: 1200, 800  ---------->    2000
  North    1200                                                                    region  amount
  South    1500      groupby       South: 1500, 600  ---------->    2100   -->      North    2000
  North     800      -------->                                                      South    2100
  South     600                    East : 950        ---------->     950            East      950
  East      950
```

**Yaad rakho:** tumhein sirf "Apply" step likhna is. Split and Combine pandas khud does is.
Interview me if poochein "groupby how work does is?" — yehi 3 lafz bolo: split, apply, combine.

In [ ]:
# Hamara main dataset — chhota is taake every output aankh with verify be sake.
# know boojh do 2 "gande" points daale are:
#   - index 9 par amount = NaN  (count vs size of farq dikhane for)
#   - index 17 par region = None (NaN group dikhane for)
orders = pd.DataFrame({
    "order_id": range(1, 19),
    "region": ["North", "North", "South", "South", "East", "East", "West", "West",
               "North", "South", "East", "West", "North", "South", "East", "West",
               "North", None],
    "channel": ["Web", "Store", "Web", "Partner", "Web", "Store", "Web", "Partner",
                "Web", "Web", "Partner", "Store", "Store", "Web", "Web", "Web",
                "Partner", "Web"],
    "rep": ["Ali", "Sara", "Ali", "Bilal", "Sara", "Ali", "Bilal", "Sara",
            "Ali", "Sara", "Bilal", "Ali", "Sara", "Bilal", "Ali", "Sara",
            "Bilal", "Ali"],
    "amount": [1200.0, 800.0, 1500.0, 600.0, 950.0, 1100.0, 2000.0, 450.0,
               1300.0, np.nan, 700.0, 1650.0, 500.0, 1900.0, 1250.0, 300.0,
               875.0, 1000.0],
    "units": [3, 2, 5, 1, 2, 4, 6, 1, 3, 2, 2, 5, 1, 7, 3, 1, 2, 2],
    "order_date": pd.to_datetime([
        "2024-01-05", "2024-01-11", "2024-01-19", "2024-01-27",
        "2024-02-02", "2024-02-08", "2024-02-14", "2024-02-20",
        "2024-02-26", "2024-03-03", "2024-03-07", "2024-03-12",
        "2024-03-15", "2024-03-19", "2024-03-22", "2024-03-25",
        "2024-03-28", "2024-03-30"]),
})

print(orders.head(6))
print("\nshape:", orders.shape)
print("\ndtypes:\n", orders.dtypes)   # NOTE: pandas 3 me text columns of dtype 'str' is, 'object' no

## 2. `groupby` object — this LAZY is

**what is:** `df.groupby("col")` ek `DataFrameGroupBy` object gives is — this **abhi some compute no does**.
this sirf plan creates is ("who row kis group me jayegi"). Compute then is when tum aggregation maangte be.

**why important:** interview me poocha goes is "groupby turant work does is?" — jawab: **no, lazy is**.
Isi wajah with `df.groupby("a")` print karne par tumhein data no, object of repr gets is.

In [ ]:
g = orders.groupby("region")     # sirf plan make — koi calculation no hui

print("type:", type(g))                  # DataFrameGroupBy
print("ngroups:", g.ngroups)             # kitne groups bane (NaN group by default count no is)
print("\n.groups (dict: key -> row labels):")
for key, idx in g.groups.items():        # .groups ek mapping is: group key -> index labels
    print(f"  {key!r:>8} -> {list(idx)}")

# NOTE: region me ek None tha (index 17) — wo group me no aaya, kyunki dropna=True default is
print("\nkul rows:", len(orders), "| groups me rows:", sum(len(i) for i in g.groups.values()))

In [ ]:
# get_group() — ek group of poora DataFrame nikalo
south = g.get_group("South")
print("South group:\n", south[["order_id", "region", "amount"]])

# Iteration — (key, sub_dataframe) tuples milte are. Debugging for behtareen.
print("\n--- iteration ---")
for key, sub in g:
    print(f"{key:>6}: rows={len(sub):>2}, amount_sum={sub['amount'].sum():>8.1f}")

# Gotcha: multiple columns par group do to key ek tuple is is
for key, sub in orders.groupby(["region", "channel"]):
    print("tuple key example:", key, "-> rows:", len(sub))
    break   # sirf pehla dikhaya

## 3. Grouping of tareeqe (7 tareeqe)

`groupby()` to tum this all can be:

| method | Syntax | when use perform |
|---|---|---|
| Single column | `df.groupby("region")` | all with aam |
| Multiple columns | `df.groupby(["region","channel"])` | 2-level breakdown → MultiIndex result |
| Series / array | `df.groupby(arr)` | key data of bahar with aa  be |
| Index level | `df.groupby(level="region")` | key index me be |
| function | `df.groupby(len)` | index par calculation with key bane |
| dict / mapping | `df.groupby({"a":"X"})` | manual bucketing (region → zone) |
| `pd.Grouper` | `pd.Grouper(key="order_date", freq="ME")` | time-based grouping (monthly, weekly) |

In [ ]:
# (a) Single column
print("(a) single column:\n", orders.groupby("region")["amount"].sum(), sep="")

# (b) Multiple columns -> MultiIndex (region, channel)
multi = orders.groupby(["region", "channel"])["amount"].sum()
print("\n(b) multiple columns (pehli 6 rows):\n", multi.head(6), sep="")
print("index nlevels:", multi.index.nlevels)   # 2 levels

# (c) Series/array with grouping — data of bahar of key
# here ek derived key make  are: amount 1000 with bara ya chhota
size_key = np.where(orders["amount"] >= 1000, "big", "small")   # numpy array
print("\n(c) array group by:\n", orders.groupby(size_key)["order_id"].count(), sep="")

In [ ]:
# (d) Index level par grouping — pehle region to index create are
by_idx = orders.set_index("region")
print("(d) level of naam with:\n", by_idx.groupby(level="region")["amount"].sum(), sep="")

# (e) function with grouping — function to INDEX of every value milti is
# here region of pehle harf par group doing are (N, S, E, W)
# Pehle region of NaN row hata di, warna function to None milega and None[0] crash karega
by_idx2 = orders.dropna(subset=["region"]).set_index("region")
print("\n(e) function (pehla harf):\n",
      by_idx2.groupby(lambda label: label[0])["amount"].sum(), sep="")

# (f) dict / mapping — business bucketing (region -> zone)
zone_map = {"North": "Upper", "South": "Lower", "East": "Upper", "West": "Lower"}
print("\n(f) dict mapping (zone):\n",
      by_idx2.groupby(zone_map)["amount"].sum(), sep="")

In [ ]:
# (g) pd.Grouper — time based grouping. freq="ME" = Month End.
# pandas 3 me freq codes: D B W ME MS QE QS this YS h min s  (purana "M"/"H" error gives is!)
monthly = orders.groupby(pd.Grouper(key="order_date", freq="ME"))["amount"].sum()
print("(g) monthly (ME = month end labels):\n", monthly, sep="")

# Purane freq codes ab error dete are — this pandas 3 of bara change is
try:
    orders.groupby(pd.Grouper(key="order_date", freq="M"))["amount"].sum()
except exception as e:
    print("\n'M' of natija ->", type(e).__name__, ":", str(e)[:90])

## 4. necessary groupby flags: `sort`, `dropna`, `observed`, `as_index`

this 4 flags 90% "arey this output aisa why is?" which sawalon of jawab are.

- **`sort=True`** (default) — group keys sorted order me comes are. `sort=False` **tez** is (bare data par farq appears is)
  and groups **first-appearance** order me come are.
- **`dropna=True`** (default) — key me NaN which rows **poori type gayab** be jati are. `dropna=False` their apna NaN group creates is.
  **this silent data loss of all with aam sabab is.**
- **`observed=True`** (pandas 3 me default) — Categorical keys par sirf **mojood** categories comes are. `observed=False` un-used
  categories to also zero of sath dikhata is.
- **`as_index=True`** (default) — group keys result of **index** bante are. `as_index=False` them normal columns contains is
  (yani flat table — merge/plot for behtar).

In [ ]:
# sort=True vs sort=False — order of farq
print("sort=True  (default, alphabetical):")
print(orders.groupby("region", sort=True)["amount"].sum())

print("\nsort=False (first appearance order — North, South, East, West):")
print(orders.groupby("region", sort=False)["amount"].sum())

In [ ]:
# dropna — SILENT DATA LOSS of demo. index 17 of region None thi.
d_true = orders.groupby("region", dropna=True)["amount"].sum()
d_false = orders.groupby("region", dropna=False)["amount"].sum()

print("dropna=True  -> groups:", d_true.shape[0], "| kul amount:", d_true.sum())
print("dropna=False -> groups:", d_false.shape[0], "| kul amount:", d_false.sum())
print("\ndropna=False of natija (see aakhir me NaN group):")
print(d_false)

# Verify: 1000.0 which row (order_id 18) dropna=True me GAYAB is
print("\ngayab hua amount:", d_false.sum() - d_true.sum(), "<- this order_id 18 is")

In [ ]:
# observed — Categorical of sath. Ek category ('Central') know boojh do khali rakhi is.
cat_regions = pd.Categorical(
    orders["region"],
    categories=["North", "South", "East", "West", "Central"],   # Central kisi row me no
)
tmp = orders.assign(region_cat=cat_regions)

obs_true = tmp.groupby("region_cat", observed=True)["amount"].sum()
obs_false = tmp.groupby("region_cat", observed=False)["amount"].sum()

print("observed=True (pandas 3 default) — sirf mojood categories:\n", obs_true, sep="")
print("\nobserved=False — un-used 'Central' also 0.0 of sath aa gaya:\n", obs_false, sep="")

In [ ]:
# as_index=False -> flat table (keys column rehte are). this merge for ideal is.
nested = orders.groupby(["region", "channel"])["amount"].sum()
flat = orders.groupby(["region", "channel"], as_index=False)["amount"].sum()

print("as_index=True  -> Series with MultiIndex, nlevels =", nested.index.nlevels)
print("as_index=False -> DataFrame, columns =", list(flat.columns))
print("\nflat table (pehli 5):\n", flat.head(5), sep="")

# Wahi work reset_index() with also is — both barabar are
same = nested.reset_index()
print("\nreset_index() of natija barabar is?", flat.equals(same))

## 5. Aggregation — single, list, dict, named

Aggregation = every group to **ek value** me sikorna.

Char shakal:

1. **Single** — `g["amount"].sum()`
2. **list** — `g["amount"].agg(["sum", "mean"])` → every func of apna column
3. **dict** — `g.agg({"amount": "sum", "units": "mean"})` → every column of apna func
4. **Named aggregation** (BEST) — `g.agg(total=("amount", "sum"), n=("order_id", "count"))`
   → saaf column names, koi MultiIndex column no. **Production me yehi use do.**

In [ ]:
g = orders.groupby("region")

# 1) Single aggregation -> Series
print("1) single:\n", g["amount"].sum(), sep="")

# 2) list of functions -> DataFrame (columns = func names)
print("\n2) list of funcs:\n", g["amount"].agg(["sum", "mean", "count", "max"]), sep="")

In [ ]:
# 3) dict — every column of alag func
print("3) dict per column:")
print(g.agg({"amount": "sum", "units": "mean", "order_id": "count"}))

# dict + list -> MultiIndex columns ban jate are (that phir dukh gives is)
multi_cols = g.agg({"amount": ["sum", "mean"], "units": "max"})
print("\ndict + list -> MultiIndex columns:")
print(multi_cols)
print("\ncolumns:", multi_cols.columns.tolist())    # tuples! ('amount','sum') ...

In [ ]:
# 4) NAMED AGGREGATION — saaf, flat, production-ready. Format: naya_naam=(column, func)
summary = orders.groupby("region").agg(
    total_revenue=("amount", "sum"),
    avg_order=("amount", "mean"),
    n_orders=("order_id", "count"),
    total_units=("units", "sum"),
    unique_reps=("rep", "nunique"),
    biggest=("amount", "max"),
)
print(summary)
print("\ncolumns flat are:", summary.columns.tolist())

# AOV (Average Order value) — business metric, derived column
summary["aov"] = (summary["total_revenue"] / summary["n_orders"]).round(2)
print("\nAOV of sath:\n", summary[["total_revenue", "n_orders", "aov"]], sep="")

In [ ]:
# Custom functions — lambda ya named func, both chalte are
def spread(s):
    """max - min: group inside amount of phailaav."""
    return s.max() - s.min()

res = orders.groupby("region")["amount"].agg([
    "mean",
    spread,                                  # named function -> column name = 'spread'
    ("range_pct", lambda s: (s.max() - s.min()) / s.max() * 100),   # (naam, func) tuple
])
print(res.round(2))

# Named aggregation of sath custom func
print("\nnamed + custom:")
print(orders.groupby("region").agg(
    spread=("amount", spread),
    p90=("amount", lambda s: s.quantile(0.90)),
).round(2))